In [ ]:
import csv #biblioteca para leitura de csv
import difflib # 
import re #tokentização esta sendo feito por meio do re
import nltk #serve para o mesmo proposito usando word_tokenize e sent_tokenize  
from unidecode import unidecode

def pre_processamento(texto):
  
    # seleciona apenas letras e coloca todas em minúsculo 
    #efetua o processo de filtragem, de letras de A a Z, com acentuação e de numero de 1 a 9
    letras_min =  re.findall(r'\b[A-zÀ-úü0-9]+\b', texto.lower())

    print(letras_min)

    # remove stopwords
    # remove complementos dentro do portugues, como e, de, para, com, um, uma etc...
    stopwords = nltk.corpus.stopwords.words('portuguese')

    #minha_stopwords = ['um', 'uma', 'varios'] #lista personalizavel
    
    #converter em uma lista as mesmas palavras não usadas anteriores
    stop = set(stopwords)

    #verificação da palavras como uma lista de compreenção
    sem_stopwords = [w for w in letras_min if w not in stop]

    print(stopwords)
    print(stop)
    print(sem_stopwords)

    # juntando os tokens novamente em formato de texto e isso precisa mudar
    texto_limpo = " ".join(sem_stopwords)

    return texto_limpo

#função para abrir o arquivo de iniciação e aplicar tudo dentro de um dicionario respostas={}
def carregar_respostas(arquivo): #função para carregar e abrir o arquivo
    respostas = {} #toda resposta vai virar parte do dicionario

    with open(arquivo, 'r', encoding='utf-8') as n: #abre o arquivo que precisa e fecha quando solicitado ou solucionado
        leitor = csv.DictReader(n) #cada linha vira um dicionario pergunta: "oi"; resposta: "tudo bem?"

        for linha in leitor: #vai fazer a leitura de cada linha
            # remover os espaços extras e transformar todas as letras em lowerscale (minusculo)
            pergunta = linha['pergunta'].strip().lower() 
            resposta = linha['resposta'].strip()
            respostas[pergunta] = resposta #copula como um dicionario
    #print(respostas) de teste
    return respostas

def carregar_problemas(arquivo): 
    problemas = {} 

    with open(arquivo, 'r', encoding='utf-8') as n: 
        leitor = csv.DictReader(n) #

        for linha in leitor: 
            pergunta = linha['pergunta'].strip().lower() 
            problema = linha['problema'].strip()
            problemas[pergunta] = problema 
    return problemas


#detectar a intenção e padrão de comunicação do usuario com o regex
def detectar_intencao(mensagem):
    mensagem_limpa = unidecode(mensagem.lower().strip())
    
    # Padrões regex para detectar intenções
    padroes = {
        'saudacao': [
            r'^o+i+!*$',                    # oi, oii, oiii, oi!
            r'^o+l+a+!*$',                  # ola, olaa, ola!
            r'^(e+\s*)?a+e+!*$',           # ae, e ae, e aeee!
            r'^o+p+a+!*$',                  # opa, opaa
            r'^f+a+l+a+!*$',                # fala, falaa
            r'^(hey|hello|hi)+!*$',         # hey, hello, hi
            r'^iai+!*$',                    # iai, iaii
            r'^[bs]om\s*dia!*$',           # bom dia, bom dia!
            r'^[bs]oa\s*tarde!*$',         # boa tarde
            r'^[bs]oa\s*noite!*$'          # boa noite
        ],
        'despedida': [
            r'^t+c+h+a+u+!*$',              # tchau, tchauu
            r'^a+t+[ée]\s*(l[oó]g[o0]|m[aá]is|j[aá])!*$', # ate logo, ate mais, ate ja
            r'^f+l+w+!*$',                  # flw, flww
            r'^(bye|goodbye)+!*$',          # bye, goodbye
            r'^v+a+l+e+u+!*$'               # valeu, valeeu
        ],
        'nome': [
            r'.*[qk]ual\s*[ée]\s*seu\s*nome.*',
            r'.*[ck]omo\s*vo[cc][ea]\s*se\s*chama.*',
            r'.*seu\s*nome.*',
            r'.*nome.*vo[cc][ea].*'
        ],
        'tudo_bem': [
            r'.*tudo\s*[bc]em.*',
            r'.*como\s*vo[cc][ea]\s*esta.*',
            r'.*como\s*vo[cc][ea]\s*t[aá].*',
            r'^[bc]e?le?za+!*$'             # blz, beleza, belezaa
        ]
    }
    
    # Verifica cada categoria
    for intencao, lista_padroes in padroes.items():
        for padrao in lista_padroes:
            if re.search(padrao, mensagem_limpa, re.IGNORECASE):
                return intencao
    
    return "outro"

#função para responder de acordo com a pergunta
def responder(mensagem, respostas): 

    #tokentização incompleta ainda
    texto = pre_processamento(mensagem)

    # Primeiro tenta detectar a intenção com regex
    intencao = detectar_intencao(texto)
    
    # Mapeia a intenção para a resposta correspondente
    # comentario.: ja temos um função carregar_repostas que retornar a mesma informação e ele puxo tudo do arquivo, poderiamos apenas pegar o retorno da função para usar como mapeamento
    mapeamento_respostas = {
        'saudacao': respostas.get('oi', 'Olá! Tudo bem?'),
        'despedida': respostas.get('tchau', 'Até logo! Volte sempre!'),
        'nome': respostas.get('qual e o seu nome', 'Eu sou um chatbot feito em Python!'),
        'tudo_bem': respostas.get('tudo bem', 'Que bom! Como posso te ajudar?')
    }

    #print(mapeamento_respostas) de teste
    
    # Se encontrou uma intenção conhecida, retorna a resposta correspondente
    # comentario.: se isso tem um if para mapear respostas, poderia ter um elif para procurar os problemas tambem e por um um else para caso não tenha nada na base de dados
    if intencao in mapeamento_respostas:
        return mapeamento_respostas[intencao]
    
    # Se não encontrou por regex, usa o sistema antigo de similaridade
    # comentario.: se conseguir mesclar o regex com o codigo legado, daqui para baixo não precisa mais usar
    mensagem_limpa = unidecode(mensagem.lower().strip())
    perguntas = list(respostas.keys())
    
    # Busca exata
    if mensagem_limpa in perguntas:
        return respostas[mensagem_limpa]
    
    # Busca por similaridade
    parecidas = difflib.get_close_matches(mensagem_limpa, perguntas, n=1, cutoff=0.6) #se tornou obsoleto
    
    if parecidas:
        return respostas[parecidas[0]]
    else:
        return "Desculpe, não entendi o que você quis dizer."



In [5]:
# Carrega as respostas do arquivo CSV
respostas = carregar_respostas("data/iniciacao.csv")
#problemas = carregar_problemas("data/problemas.csv") ainda esta incompleto, mas o escopo inicial foi aplicado

#vieira precisa fazer o codigo carregar os arquivo apenas quando for soliciado de acordo com a pergunta
    
print("ChatBot: Olá! Digite 'sair' para encerrar.\n")
    
while True:
    usuario = input("Você: ")
    if usuario.lower() == "sair":
        print("ChatBot: Até mais!")
        break
    print("ChatBot:", responder(usuario, respostas))

ChatBot: Olá! Digite 'sair' para encerrar.

['oi']
['a', 'à', 'ao', 'aos', 'aquela', 'aquelas', 'aquele', 'aqueles', 'aquilo', 'as', 'às', 'até', 'com', 'como', 'da', 'das', 'de', 'dela', 'delas', 'dele', 'deles', 'depois', 'do', 'dos', 'e', 'é', 'ela', 'elas', 'ele', 'eles', 'em', 'entre', 'era', 'eram', 'éramos', 'essa', 'essas', 'esse', 'esses', 'esta', 'está', 'estamos', 'estão', 'estar', 'estas', 'estava', 'estavam', 'estávamos', 'este', 'esteja', 'estejam', 'estejamos', 'estes', 'esteve', 'estive', 'estivemos', 'estiver', 'estivera', 'estiveram', 'estivéramos', 'estiverem', 'estivermos', 'estivesse', 'estivessem', 'estivéssemos', 'estou', 'eu', 'foi', 'fomos', 'for', 'fora', 'foram', 'fôramos', 'forem', 'formos', 'fosse', 'fossem', 'fôssemos', 'fui', 'há', 'haja', 'hajam', 'hajamos', 'hão', 'havemos', 'haver', 'hei', 'houve', 'houvemos', 'houver', 'houvera', 'houverá', 'houveram', 'houvéramos', 'houverão', 'houverei', 'houverem', 'houveremos', 'houveria', 'houveriam', 'houveríamo